# Avaliacao de Multi Layer Perceptron
## Imports e funções

In [ ]:
import pandas as pd
import numpy as np
from numpy import mean
from numpy import std
from sklearn import metrics
from sklearn.metrics import confusion_matrix, f1_score

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

from sklearn.neural_network import MLPClassifier

def raca_para_especie(raca):
    if raca in ['basset_hound', 'saint_bernard']:
        return 'dog'
    elif raca in ['Birman', 'Persian']:
        return 'cat'
    else:
        return raca  # fallback

### Para Bases com PCA que geram valores negativos
from sklearn.preprocessing import minmax_scale

def separar_dataset(df, scale=False):
    X = df.iloc[:, :-1]
    if scale:
        X = minmax_scale(X)
    y = df.iloc[:, -1]
    return X, y




## Lendo lista de arquivos a processar

In [3]:
datafiles = pd.read_csv('dataset_list.csv',encoding='utf-8')

datafiles.head(12)

,key,filename
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz


## Lendo Dataframes e ajustando dados

In [27]:

dfs = {}
shapes = []
last_cols = []

for metadata in datafiles.itertuples():
    # Imprime o arquivo que está sendo lido
    #print(f"Lendo arquivo: {metadata.key}")
    # Carrega o DataFrame
    df = pd.read_csv(metadata.filename)
    
    # Aplica a função raca_para_especie na coluna raca
    if 'raca' in df.columns:
        df['especie'] = df['raca'].apply(raca_para_especie)
        df = df.drop('raca', axis=1)
    
    # Elimina a coluna nome_arquivo se existir
    if 'nome_arquivo' in df.columns:
        df = df.drop('nome_arquivo', axis=1)
    
    dfs[metadata.key] = df
    shapes.append(df.shape)
    # Pega as últimas duas colunas
    last_cols.append(df.columns[-1:].tolist())

# Adiciona as colunas shape e last_two_columns ao datafiles
datafiles['shape'] = shapes
datafiles['last_column'] = last_cols

datafiles
    

,key,filename,shape,last_column
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz,"(800, 104)",[especie]
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz,"(800, 649)",[especie]
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz,"(800, 325)",[especie]
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz,"(800, 3601)",[especie]
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz,"(800, 94)",[especie]
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz,"(800, 118)",[especie]
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz,"(800, 325)",[especie]
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz,"(800, 51)",[especie]
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz,"(800, 99)",[especie]
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz,"(800, 1765)",[especie]


## Identificando as melhores 10 configurações a partir da melhor base de dados disponível

In [42]:
dataframe_zero = dfs[datafiles['key'][0]]

param_grid = {
    'hidden_layer_sizes': [(50), (100), (120), (150), (70,70), (100,50)],
    'activation': ['identity', 'logistic', 'tanh', 'relu'],
    'solver': ['adam', 'sgd'],
    'learning_rate_init': [0.0001, 0.001, 0.01, 0.1],
    'max_iter': [500, 1000, 1500, 2000]
}

X, y = separar_dataset(dataframe_zero, scale=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

# Grid search para encontrar top 10 configurações
from sklearn.model_selection import ParameterGrid
from sklearn.exceptions import ConvergenceWarning

print("Executando grid search com holdout...")
results_holdout = []
erros_encontrados = 0
warnings_encontrados = 0
sucessos = 0

# Gerando todas as combinações de parâmetros
param_combinations = list(ParameterGrid(param_grid))
print(f"Total de combinações a testar: {len(param_combinations)}")

for i, params in enumerate(param_combinations):
    print(f"Testando configuração {i+1}/{len(param_combinations)}: {params}")
    
    try:
        # Criando e treinando o modelo
        mlp = MLPClassifier(**params, random_state=42)
        
        # Capturando warnings durante o treinamento
        import warnings
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            mlp.fit(X_train, y_train)
            
            # Verificando se houve warnings de convergência
            convergence_warning = any(issubclass(warning.category, ConvergenceWarning) for warning in w)
            
        # Avaliando no conjunto de teste
        y_pred = mlp.predict(X_test)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        if convergence_warning:
            warnings_encontrados += 1
            print(f"\t⚠ WARNING: Não convergiu - F1 Score: {f1:.4f}")
            # Não adiciona às top 10 se teve warning de convergência
        else:
            sucessos += 1
            print(f"\t✓ Sucesso - F1 Score: {f1:.4f}")
            results_holdout.append({
                'params': params,
                'f1_score': f1,
                'status': 'sucesso'
            })
        
    except (ValueError, ConvergenceWarning) as e:
        erros_encontrados += 1
        print(f"\t✗ ERRO: {type(e).__name__}")
        continue
    except Exception as e:
        erros_encontrados += 1
        print(f"\t✗ ERRO INESPERADO: {type(e).__name__}")
        continue

# Ordenando por F1 score
results_df = pd.DataFrame(results_holdout)
results_df = results_df.sort_values('f1_score', ascending=False)

# Selecionando as top 10 configurações
top_10_configs = results_df.head(10).copy()
top_10_configs.reset_index(drop=True, inplace=True)

print("\nTop 10 configurações encontradas:")
print(top_10_configs)

# Salvando as top 10 configurações para uso posterior
top_10_params = top_10_configs['params'].tolist()

print(f"\n" + "="*60)
print("RELATÓRIO FINAL DO GRID SEARCH")
print("="*60)
print(f"Total de combinações testadas: {len(param_combinations)}")
print(f"✓ Sucessos (sem warnings): {sucessos}")
print(f"⚠ Warnings (não convergiu): {warnings_encontrados}")
print(f"✗ Erros: {erros_encontrados}")
print(f"Top 10 configurações válidas: {len(top_10_params)}")
print("="*60)


Executando grid search com holdout...
Total de combinações a testar: 768
Testando configuração 1/768: {'activation': 'identity', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 500, 'solver': 'adam'}
	✓ Sucesso - F1 Score: 0.7647
Testando configuração 2/768: {'activation': 'identity', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 500, 'solver': 'sgd'}
	✓ Sucesso - F1 Score: 0.7530
Testando configuração 3/768: {'activation': 'identity', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 1000, 'solver': 'adam'}
	✓ Sucesso - F1 Score: 0.7647
Testando configuração 4/768: {'activation': 'identity', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 1000, 'solver': 'sgd'}
	✓ Sucesso - F1 Score: 0.7530
Testando configuração 5/768: {'activation': 'identity', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 1500, 'solver': 'adam'}
	✓ Sucesso - F1 Score: 0.7647
Testando configuração 6/768: {'activation

## TOP 10 configurações identificadas

In [50]:
from IPython.display import display

print("\nTop 10 configurações encontradas:")

# Mostra apenas as colunas 'params' e 'f1_score' (ocultando 'status')
cols_to_show = [col for col in top_10_configs.columns if col != 'status']

with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(top_10_configs[cols_to_show])



Top 10 configurações encontradas:


,params,f1_score
0,"{'activation': 'tanh', 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.0001, 'max_iter': 2000, 'solver': 'sgd'}",0.826167
1,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 1000, 'solver': 'sgd'}",0.815108
2,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 1500, 'solver': 'sgd'}",0.815108
3,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 2000, 'solver': 'sgd'}",0.815108
4,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'sgd'}",0.815108
5,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.001, 'max_iter': 2000, 'solver': 'sgd'}",0.815108
6,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.001, 'max_iter': 1500, 'solver': 'sgd'}",0.815108
7,"{'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.001, 'max_iter': 1000, 'solver': 'sgd'}",0.815108
8,"{'activation': 'logistic', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.0001, 'max_iter': 2000, 'solver': 'sgd'}",0.802370
9,"{'activation': 'logistic', 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.1, 'max_iter': 2000, 'solver': 'sgd'}",0.802370


## Aplicando as top10 configurações nas 12 bases de dados

In [52]:
# Aplicando top 10 configurações MLP em todas as bases de dados
par_training = ["holdout", "crossvalidation"]

# Estrutura para armazenar resultados
resultados_mlp = []

print("Aplicando top 10 configurações MLP em todas as bases de dados...")
print("=" * 70)

for metadata in datafiles.itertuples():
    print(f"Dataset: {metadata.key}")
    
    # Obtém o dataset correspondente do dicionário dfs
    df = dfs[metadata.key]
    X, y = separar_dataset(df, False)
    
    # Testando cada uma das top 10 configurações
    for i, params in enumerate(top_10_params):
        print(f"\tConfiguração {i+1}/10: {params}")
        
        for training in par_training:
            print(f"\t\tTraining: {training}")
            
            try:
                # Criando o modelo com os parâmetros específicos
                mlp = MLPClassifier(**params, random_state=42)
                f1 = 0.0
                f1_std = 0.0
                confusao = np.array([])
                
                if training == "holdout":
                    # Treinamento e avaliação usando holdout
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
                    
                    # Capturando warnings durante o treinamento
                    import warnings
                    with warnings.catch_warnings(record=True) as w:
                        warnings.simplefilter("always")
                        mlp.fit(X_train, y_train)
                        
                        # Verificando se houve warnings de convergência
                        convergence_warning = any(issubclass(warning.category, ConvergenceWarning) for warning in w)
                    
                    y_pred = mlp.predict(X_test)
                    f1 = f1_score(y_test, y_pred, average='weighted')
                    f1_std = 0.0
                    confusao = confusion_matrix(y_test, y_pred)
                    
                    if convergence_warning:
                        print(f"\t\t\t⚠ WARNING: Não convergiu - F1: {f1:.4f}")
                    else:
                        print(f"\t\t\t✓ Sucesso - F1: {f1:.4f}")
                        
                elif training == "crossvalidation":
                    # Treinamento e avaliação usando crossvalidation
                    kf = KFold(n_splits=10, random_state=42, shuffle=True)
                    
                    # Capturando warnings durante o treinamento
                    import warnings
                    with warnings.catch_warnings(record=True) as w:
                        warnings.simplefilter("always")
                        scores = cross_val_score(mlp, X, y, scoring='f1_weighted', cv=kf)
                        y_pred = cross_val_predict(mlp, X, y, cv=kf)
                        
                        # Verificando se houve warnings de convergência
                        convergence_warning = any(issubclass(warning.category, ConvergenceWarning) for warning in w)
                    
                    confusao = confusion_matrix(y, y_pred)
                    f1 = scores.mean()
                    f1_std = scores.std()
                    
                    if convergence_warning:
                        print(f"\t\t\t⚠ WARNING: Não convergiu - F1: {f1:.4f} ({f1_std:.4f})")
                    else:
                        print(f"\t\t\t✓ Sucesso - F1: {f1:.4f} ({f1_std:.4f})")
                else:
                    print(f"\t\t\tTraining type {training} nao suportado")
                    continue
                
                # Armazena resultados no DataFrame
                resultado = {
                    'dataset': metadata.key,
                    'config_rank': i + 1,
                    'params': params,
                    'training_type': training,
                    'f1_score': f1,
                    'f1_std': f1_std,
                    'confusion_matrix': confusao,
                    'model': mlp
                }
                resultados_mlp.append(resultado)
                
            except Exception as e:
                print(f"\t\t\t✗ ERRO: {type(e).__name__}")
                continue
    
    print("-" * 50)

# Converte para DataFrame
df_resultados_mlp = pd.DataFrame(resultados_mlp)
                
df_resultados_mlp['shape'] = df_resultados_mlp['dataset'].apply(lambda x: datafiles.loc[datafiles['key'] == x, 'shape'].values[0])



Aplicando top 10 configurações MLP em todas as bases de dados...
Dataset: hogfeat_128_16_4_9_pca
	Configuração 1/10: {'activation': 'tanh', 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.0001, 'max_iter': 2000, 'solver': 'sgd'}
		Training: holdout
			✓ Sucesso - F1: 0.7632
		Training: crossvalidation
			✓ Sucesso - F1: 0.7462 (0.0398)
	Configuração 2/10: {'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 1000, 'solver': 'sgd'}
		Training: holdout
			✓ Sucesso - F1: 0.7548
		Training: crossvalidation
			✓ Sucesso - F1: 0.7702 (0.0251)
	Configuração 3/10: {'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 1500, 'solver': 'sgd'}
		Training: holdout
			✓ Sucesso - F1: 0.7548
		Training: crossvalidation
			✓ Sucesso - F1: 0.7702 (0.0251)
	Configuração 4/10: {'activation': 'tanh', 'hidden_layer_sizes': 100, 'learning_rate_init': 0.01, 'max_iter': 2000, 'solver': 'sgd'}
		Training: holdout
			✓ Sucesso - F

## Salvando resultados em CSV e Joblib

In [54]:
# Salvando resultados MLP
import joblib
joblib.dump(df_resultados_mlp, 'mlp_avaliacao_resultados.joblib')

df_csv_mlp = df_resultados_mlp.drop(['confusion_matrix', 'model'], axis=1)
df_csv_mlp.to_csv('mlp_avaliacao_resultados.csv', index=False)

print(f"Resultados MLP salvos:")
print(f"- mlp_avaliacao_resultados.csv: {len(df_csv_mlp)} registros")
print(f"- mlp_avaliacao_resultados.joblib: dados completos com modelos")

Resultados MLP salvos:
- mlp_avaliacao_resultados.csv: 240 registros
- mlp_avaliacao_resultados.joblib: dados completos com modelos


### Amostra do DataFrame de Resultados

In [55]:
# Visualiza os resultados MLP
print(f"Total de combinações processadas: {len(df_resultados_mlp)}")
print(f"Colunas disponíveis: {df_resultados_mlp.columns.tolist()}")
print(f"Datasets processados: {df_resultados_mlp['dataset'].nunique()}")
print(f"Configurações por dataset: {df_resultados_mlp['config_rank'].nunique()}")

# Análise dos melhores resultados por dataset
melhores_por_dataset = df_resultados_mlp.loc[df_resultados_mlp.groupby('dataset')['f1_score'].idxmax()]
print(f"\nMelhores F1 scores por dataset:")
for _, row in melhores_por_dataset.iterrows():
    print(f"  {row['dataset']}: {row['f1_score']:.4f} (Config {row['config_rank']}, {row['training_type']})")

df_resultados_mlp


Total de combinações processadas: 240
Colunas disponíveis: ['dataset', 'config_rank', 'params', 'training_type', 'f1_score', 'f1_std', 'confusion_matrix', 'model', 'shape']
Datasets processados: 12
Configurações por dataset: 10

Melhores F1 scores por dataset:
  hogfeat_128_16_2_9: 0.8131 (Config 10, holdout)
  hogfeat_128_16_2_9_pca: 0.7876 (Config 6, holdout)
  hogfeat_128_16_4_9: 0.8048 (Config 8, holdout)
  hogfeat_128_16_4_9_pca: 0.7764 (Config 10, crossvalidation)
  hogfeat_128_32_2_9: 0.7653 (Config 6, crossvalidation)
  hogfeat_256_32_2_9: 0.7926 (Config 1, crossvalidation)
  hogfeat_256_32_2_9_pca: 0.8030 (Config 10, crossvalidation)
  hogfeat_256_64_2_18: 0.8001 (Config 8, crossvalidation)
  hogfeat_256_64_2_9: 0.7788 (Config 6, crossvalidation)
  lbpfeat_256_12_96: 0.6884 (Config 2, holdout)
  lbpfeat_256_24_192: 0.6967 (Config 2, holdout)
  lbpfeat_256_6_48: 0.6634 (Config 5, crossvalidation)


,dataset,config_rank,params,training_type,f1_score,f1_std,confusion_matrix,model,shape
0,hogfeat_128_16_4_9_pca,1,"{'activation': 'tanh', 'hidden_layer_sizes': (...",holdout,0.763210,0.000000,"[[82, 24], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
1,hogfeat_128_16_4_9_pca,1,"{'activation': 'tanh', 'hidden_layer_sizes': (...",crossvalidation,0.746158,0.039768,"[[287, 113], [90, 310]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
2,hogfeat_128_16_4_9_pca,2,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",holdout,0.754799,0.000000,"[[80, 26], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
3,hogfeat_128_16_4_9_pca,2,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",crossvalidation,0.770159,0.025076,"[[307, 93], [91, 309]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
4,hogfeat_128_16_4_9_pca,3,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",holdout,0.754799,0.000000,"[[80, 26], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
...,...,...,...,...,...,...,...,...,...
235,lbpfeat_256_24_192,8,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",crossvalidation,0.320970,0.108902,"[[123, 277], [152, 248]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 195)"
236,lbpfeat_256_24_192,9,"{'activation': 'logistic', 'hidden_layer_sizes...",holdout,0.400089,0.000000,"[[0, 106], [0, 134]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
237,lbpfeat_256_24_192,9,"{'activation': 'logistic', 'hidden_layer_sizes...",crossvalidation,0.336109,0.076863,"[[0, 400], [0, 400]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
238,lbpfeat_256_24_192,10,"{'activation': 'logistic', 'hidden_layer_sizes...",holdout,0.270617,0.000000,"[[106, 0], [134, 0]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
